In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
import json
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def fetch_ticker_history(ticker, period="10y"):
    df = yf.Ticker(ticker).history(period=period, interval="1d").reset_index()
    df["ticker"] = ticker
    df["source"] = "yfinance"
    df["ingested_at"] = datetime.utcnow()
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df

def fetch_ticker_history_range(ticker, start_date, end_date):
    df = yf.Ticker(ticker).history(start=start_date, end=end_date, interval="1d").reset_index()
    df["ticker"] = ticker
    df["source"] = "yfinance"
    df["ingested_at"] = datetime.utcnow()
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df

def load_ticker_list(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)
        return [item["ticker"] for item in config]

In [0]:
test_df = fetch_ticker_history_range("AAPL", "2024-01-01", "2024-06-01")
test_df.head()

In [0]:
seed_tickers = load_ticker_list("../config/tickers.json")
print(seed_tickers)


In [0]:
raw_pdf = pd.concat([fetch_ticker_history(t) for t in seed_tickers],ignore_index=True)


In [0]:
raw_pdf.shape


In [0]:
raw_pdf["ticker"].value_counts()

In [0]:
raw_pdf[raw_pdf["ticker"] == "COST"]["date"].agg(["min", "max", "count"])


In [0]:
raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"].agg(["min", "max", "count"])

In [0]:
aapl_dates = set(raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"])
cost_dates = set(raw_pdf[raw_pdf["ticker"] == "COST"]["date"])
aapl_dates - cost_dates

In [0]:
raw_df = spark.createDataFrame(raw_pdf)
raw_df.createOrReplaceTempView("stg_daily_prices")

In [0]:
%sql
SELECT TICKER, COUNT(*) AS ROW_COUNT
FROM stg_daily_prices
GROUP BY TICKER
ORDER BY TICKER

In [0]:
raw_df.printSchema()

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS mashup_learning;

CREATE SCHEMA IF NOT EXISTS mashup_learning.stocks;
    
CREATE TABLE IF NOT EXISTS mashup_learning.stocks.bronze_daily_prices (
  date TIMESTAMP,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE,
  volume BIGINT,
  dividends DOUBLE,
  stock_splits DOUBLE,
  ticker STRING,
  source STRING,
  ingested_at TIMESTAMP
)
USING DELTA;


In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
SHOW TABLES IN mashup_learning.stocks

In [0]:
%sql
DESCRIBE TABLE mashup_learning.stocks.bronze_daily_prices;

In [0]:
%sql
MERGE INTO mashup_learning.stocks.bronze_daily_prices
USING stg_daily_prices
ON bronze_daily_prices.date = stg_daily_prices.date AND bronze_daily_prices.ticker = stg_daily_prices.ticker
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.bronze_daily_prices;
    

In [0]:
%sql DESCRIBE HISTORY mashup_learning.stocks.bronze_daily_prices